# Day 3 · 3교시 [실습] ML 파이프라인 — `03_ml_pipeline`

## 실습 목표

전처리→학습→평가→반복을 직접 돌리고, 무엇보다 **AI가 조용히 저지르는 함정**을
**점수로 실측**한다. ML 코드는 문법 오류가 없어도 **방법론이 틀리면** 가짜 점수를 낸다 —
그래서 AI가 짠 ML 코드는 특히 사람이 검증해야 한다(교안 3.8).

| 순서 | 내용 | 교안 |
|------|------|------|
| 1 | 데이터 로드·탐색 | 3.3 |
| 2 | 전처리(분할 먼저 → 스케일) | 3.4 |
| 3 | 학습(baseline) + train-test 격차 | 3.5 |
| 4 | 평가(혼동행렬) | 3.6 |
| 5 | 실험 반복(스윕) | 3.7 |
| 6 | **함정 ① 데이터 누수** 실측 | 3.8 |
| 7 | **함정 ② 잘못된 지표** 실측 | 3.8 |

> 전부 **오프라인**(sklearn 내장·합성 데이터), **결정적**(시드 고정 — 몇 번 돌려도 같은 숫자).
> 의존성: `pip install scikit-learn numpy`

## 1. 데이터 로드·탐색 (3.3)

탐색의 목적은 구경이 아니라 **전처리에서 무엇이 필요한지 정하는 것**이다 —
스케일 차이가 큰가? 클래스가 불균형한가? 결측이 있는가?

In [ ]:
import numpy as np
from sklearn.datasets import load_wine

wine = load_wine()
X, y = wine.data, wine.target
names = [str(n) for n in wine.target_names]

print("데이터:", X.shape, "| 클래스:", names)
print("클래스 분포:", {n: int(c) for n, c in zip(names, np.bincount(y))})
print("결측치:", int(np.isnan(X).sum()))
print(f"특성 값 범위: {X.min():.2f} ~ {X.max():.1f}   ← 스케일 차이가 크다 → 표준화 필요")

## 2. 전처리 — 분할 먼저, 스케일은 그 다음 (3.4)

교안 3.4·3.8의 철칙: **train/test 분할을 먼저** 하고, 스케일러는 **train 에만 `fit`**,
test 에는 `transform` 만. 순서를 뒤집으면 test 정보가 train 으로 새어 든다(6절에서 실측).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler().fit(Xtr)      # ← train 에만 fit
Xtr_s = scaler.transform(Xtr)
Xte_s = scaler.transform(Xte)           # ← test 는 transform 만 (fit 아님)

print("분할:", Xtr.shape[0], "train /", Xte.shape[0], "test")
print("스케일러는 train 에만 fit — test 는 '처음 보는 데이터'로 남겨 둔다")

## 3. 학습 — baseline 부터 (3.5)

복잡한 모델 전에 **단순한 기준선**. baseline 이 있어야 "복잡한 모델이 정말 나은가"를
판단할 수 있다. 그리고 정확도 하나가 아니라 **train-test 격차**를 같이 본다.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

clf = LogisticRegression(max_iter=1000, random_state=42).fit(Xtr_s, ytr)

train_acc = accuracy_score(ytr, clf.predict(Xtr_s))
test_acc = accuracy_score(yte, clf.predict(Xte_s))

print(f"baseline train acc: {train_acc:.3f}")
print(f"baseline test  acc: {test_acc:.3f}   ← 이게 '일반화 성능'이다")
print(f"train-test 격차   : {train_acc - test_acc:+.3f}   ← 크게 벌어지면 과적합 신호")

## 4. 평가 — 혼동행렬 (3.6)

정확도 하나로는 부족하다. **혼동행렬**로 *어느 클래스를 어떻게 틀리는지* 본다.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

pred = clf.predict(Xte_s)
cm = confusion_matrix(yte, pred)
w = max(len(n) for n in names)

print("혼동행렬 (행=실제, 열=예측):")
print(" " * (w + 2) + "  ".join(f"{n:>{w}}" for n in names))
for i, row in enumerate(cm):
    print(f"{names[i]:>{w}}  " + "  ".join(f"{v:>{w}}" for v in row))

print()
print(classification_report(yte, pred, target_names=names, digits=2))

## 5. 실험 반복을 위임 (3.7)

사람은 *어떤 축을 실험할지*(가설)만 정하고, 돌리고 비교하는 **손발은 맡긴다**.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

print(f"{'n_estimators':>12} | {'test acc':>8}")
print("-" * 25)
for n in [10, 50, 100, 200]:
    rf = RandomForestClassifier(n_estimators=n, random_state=42).fit(Xtr_s, ytr)
    print(f"{n:>12} | {accuracy_score(yte, rf.predict(Xte_s)):>8.3f}")

print()
print("→ 나무를 늘려도 어느 지점부터는 안 오른다. '더 크게'가 답이 아닐 때가 많다.")

## 6. 함정 ① — 데이터 누수(leakage) 실측 (3.8)

**이 절이 3교시의 핵심이다.**

일부러 **신호가 0인 데이터**를 쓴다 — 특성도 정답도 전부 난수라, 아무리 좋은 모델도
정답률은 **동전 던지기(0.5)** 여야 한다. 그런데 **특성 선택을 분할 전에** 하면
(= test 의 정답을 미리 훔쳐보면) 점수가 그럴듯하게 올라간다.

두 버전 모두 **문법 오류가 없고 잘 실행된다.** 그게 무서운 점이다.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

rng = np.random.default_rng(0)
X_noise = rng.normal(size=(200, 2000))          # 특성: 순수 난수
y_noise = rng.integers(0, 2, size=200)          # 정답: 순수 난수 → 배울 신호가 0


def evaluate(leaky: bool) -> float:
    if leaky:
        # ✗ 누수: 전체 데이터로 특성 선택 (test 정답을 봐 버린다) → 그 다음 분할
        X_sel = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)
        a, b, c, d = train_test_split(X_sel, y_noise, test_size=0.3, random_state=0)
    else:
        # ✓ 올바름: 분할 먼저 → train 에서만 특성 선택
        a, b, c, d = train_test_split(X_noise, y_noise, test_size=0.3, random_state=0)
        sel = SelectKBest(f_classif, k=20).fit(a, c)
        a, b = sel.transform(a), sel.transform(b)
    model = LogisticRegression(max_iter=1000).fit(a, c)
    return accuracy_score(d, model.predict(b))


leaky_acc = evaluate(leaky=True)
honest_acc = evaluate(leaky=False)

print("데이터: 특성도 정답도 난수 — 배울 신호가 전혀 없다")
print()
print(f"누수 있는 버전 (X): test acc = {leaky_acc:.3f}   ← 신호가 없는데 이 점수가 나온다")
print(f"올바른 버전   (O): test acc = {honest_acc:.3f}   ← 동전 던지기 수준 = 정직한 답")
print()
print(f"부풀림 +{leaky_acc - honest_acc:.3f}  —  문법 오류는 없다. '방법론'이 틀렸을 뿐이다.")

## 7. 함정 ② — 잘못된 지표 (3.8)

불균형 데이터에서 **정확도는 거짓말을 한다.** 아무것도 학습하지 않고
"전부 다수 클래스"라고만 찍는 모델의 점수를 보자.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, recall_score

rng2 = np.random.default_rng(7)          # 셀마다 독립 시드 — 실행 순서가 결과를 안 바꾼다
X_imb = rng2.normal(size=(1000, 5))
y_imb = (rng2.random(1000) < 0.02).astype(int)     # 양성 2% 안팎 (희귀 사건)

Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(
    X_imb, y_imb, test_size=0.3, random_state=0
)

dummy = DummyClassifier(strategy="most_frequent").fit(Xi_tr, yi_tr)
dp = dummy.predict(Xi_te)

print(f"양성 비율: {y_imb.mean():.1%}  (불량품·질병·이상거래 같은 희귀 사건)")
print()
print("'전부 정상'이라고만 찍는 모델의 성적표:")
print(f"  accuracy = {accuracy_score(yi_te, dp):.3f}   ← 훌륭해 보인다")
print(f"  recall   = {recall_score(yi_te, dp, zero_division=0):.3f}   ← 정작 찾아야 할 걸 하나도 못 잡았다")
print(f"  f1       = {f1_score(yi_te, dp, zero_division=0):.3f}")
print()
print("→ 무엇을 맞혀야 하는 문제인지에 따라 '봐야 할 지표'가 다르다.")

## 실습 정리

- 전처리(**분할 먼저 → 스케일**) → baseline → 평가(혼동행렬) → 스윕을 직접 돌렸다.
- **함정 ① 데이터 누수**: 신호가 0인 데이터에서 누수 버전이 `0.750`, 올바른 버전이
  `0.450`. **없는 실력이 +0.300 만큼 만들어졌다.**
- **함정 ② 잘못된 지표**: 아무것도 안 하는 모델이 `accuracy 0.983` —
  그런데 `recall 0.000`. 정확도만 보면 속는다.
- 둘 다 **실행은 완벽하다.** 문법 오류가 아니라 **방법론 오류**라서 그렇다.
  Day 1의 "미묘한 오류"가 ML 에선 이렇게 나타난다.

> **원칙**: 반복 손발은 에이전트에게, **가설과 검증은 사람이**.
> 정확도가 아니라 **일반화**로 판단하고, train/test 격차·누수·지표 선택은 사람이 지킨다.